# Grande Consultation — premiers graphiques

One chart per cell, to be tested **bit by bit**.
Each chart shows its **base** and, for simple bars, the 95 % confidence interval if enabled.

- To save the images as PNG: set `SAVE = True` in the settings cell.
- To hide the error bars: `SHOW_CI = False`.



## 0. Préparation (à exécuter une fois)

In [ ]:
from pathlib import Path
import sys

DATA_PATH = "data/20261002_gc_reponses.xlsx"  # ← questionnaire export
SAVE = False  
SHOW_CI = False  
OUTPUT_DIR = "resultats/graphiques/gc"

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
DATA_PATH, OUTPUT_DIR = ROOT / DATA_PATH, ROOT / OUTPUT_DIR

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

import charts as ch
import geo
import grande_consultation as g

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.float_format = "{:.3f}".format

df = g.load(DATA_PATH)  # raw export, repeated Submission IDs removed
qs = g.discover_questions(df)  # {question number: description}
g.check_structure(qs, df)
g.explode_text_answers(df, qs)  # one boolean column per option for the multi-choice questions
d, unmapped = g.recode(df, qs)  # clean table: scores, age bracket, profile variables
freq = g.frequency_tables(df, d, qs)  # one-way tables of the single-choice questions
multi = g.multiple_choice_tables(df, qs)  # one-way tables of the multiple-choice questions
N = len(df)
print(f"{N} répondant·es, {len(qs)} questions")
for n, vals in unmapped.items():
    print(f"⚠ Q{n} : libellés non reconnus pour le score : {vals}")

ch.SHOW_CI = SHOW_CI

In [ ]:
# one chart = one short call.
def freq_of(n, order=None):
    """Frequency table of question n; `order` (list of answers) fixes the row order."""
    t = freq[freq["question"] == n].copy()
    if order:
        t["_o"] = t["réponse"].map({v: i for i, v in enumerate(order)}).fillna(len(order))
        t = t.sort_values("_o")
    return t


def scale(n):
    """Categories of an ordinal question, from negative to positive, then the "no opinion" ones."""
    m = g.ORDINALS[n]
    return sorted(m, key=m.get) + sorted(g.NO_OPINION)


def title_of(n):
    """Question text, used as chart title."""
    return qs[n]["text"]


def single_choice_bars(n, order=None, **kw):
    """Bar chart of a single-choice question (sorted by size unless `order` is given)."""
    t = freq_of(n, order)
    ch.bars(
        t, "réponse", "%", "IC95 bas", "IC95 haut", title=title_of(n), base=t["base"].max(), sort=order is None, **kw
    )


def multiple_choice_bars(n, **kw):
    """Bar chart of a multiple-choice question (one bar per option)."""
    t = multi[multi["question"] == n]
    ch.bars(t, "option", "% des répondants", "IC95 bas", "IC95 haut", title=title_of(n), base=t["base"].max(), **kw)


def stacked_questions(ns, labels, fig_title, note=None):
    """100 % stacked bars for several questions that share the same scale (one bar per question)."""
    t = ch.long_from_tables(freq, ns, labels)
    order = []
    for n in ns:
        order += [m for m in scale(n) if m not in order]
    ch.stacked(t, "row", "category", "pct", order=order, title=fig_title, note=note)


def finish(name):
    """Save the current figure as PNG when SAVE is True, then show it.

    The saving must come BEFORE the display: the inline backend closes the figure once shown,
    and the current figure would then be a new, empty one (blank PNG).
    """
    path = ch.save_figure(plt.gcf(), name, OUTPUT_DIR, SAVE)
    plt.show()
    if path:
        print("enregistré :", path)

## 1. Qui a répondu (n° 1)


In [ ]:
fig, axes = ch.panels(4, ncols=2, width=6.4, height=3.2)
orders = {24: g.ORDERS["tranche_age"], 26: g.ORDERS["niveau_etudes"], 27: g.ORDERS["situation"]}
for ax, n in zip(axes, [24, 25, 26, 27]):
    t = freq_of(n, orders.get(n))
    ch.bars(
        t,
        "réponse",
        "%",
        "IC95 bas",
        "IC95 haut",
        title=f"Q{n} · " + qs[n]["text"].split(". ", 1)[-1],
        base=t["base"].max(),
        sort=n not in orders,
        ax=ax,
        n_max=10,
    )
plt.tight_layout()
finish("gc_01_profil")

## 2. Origine géographique (n° 2)
Départements with at least 10 answers (threshold `MIN_N_DEPT`). In orange: Île-de-France.

In [ ]:
dept = d.groupby("departement", dropna=True).agg(n=("departement", "size"), idf=("region_idf", "first")).reset_index()
dept = dept[dept["n"] >= g.MIN_N_DEPT].copy()
dept["share"] = dept["n"] / N
colours = {r.departement: ch.ORANGE for r in dept.itertuples() if r.idf == "Île-de-France"}
ch.bars(
    dept,
    "departement",
    "share",
    title="Q28 · Département de résidence",
    base=d["departement"].notna().sum(),
    note="orange = Île-de-France",
    highlight=colours,
    n_max=20,
)
finish("gc_02_departements")

### 2b. Cartes de France : réponses par département et par région
The same Q28 answers drawn on a map. The département code is read from the answer ("75 - Paris" → 75).
**Small counts are never shown**: below 10 answers (départements) or 20 (régions) a shape stays grey and unlabelled, because the data contain minors.
At département level, a zoom on Île-de-France is added (Paris and its inner ring are too small to read otherwise); overseas départements are drawn as insets.

In [ ]:
dept_counts = geo.count_by_department(d["departement"])  # answers per département code (0 when nobody answered)
n_with_dept = int(dept_counts.sum())
unknown = dept_counts.attrs["unknown"]
if unknown:
    print(f"⚠ {unknown} réponse(s) dont le département n'est pas reconnu (ignorées sur la carte)")

ch.map_france(
    dept_counts,
    level="department",
    title="Q28 · Nombre de réponses par département",
    base=n_with_dept,
    min_n=g.MIN_N_DEPT,
)
finish("gc_02b_carte_departements")

In [ ]:
ch.map_france(
    dept_counts,
    level="region",
    title="Q28 · Nombre de réponses par région",
    base=n_with_dept,
    min_n=g.MIN_N,
)
finish("gc_02c_carte_regions")

## 3. Causes proches et causes urgentes (n° 3)
Q1: causes the person feels closest to. Q2: causes the candidates should deal with first. Same list, two questions.

In [ ]:
close = multi[multi["question"] == 1].assign(series="Plus proches (Q1)")
urgent = multi[multi["question"] == 2].assign(series="Plus urgentes (Q2)")
t = pd.concat([close, urgent])
ch.grouped(
    t,
    "option",
    "series",
    "% des répondants",
    series_order=["Plus proches (Q1)", "Plus urgentes (Q2)"],
    title="Causes : plus proches de soi, ou plus urgentes à traiter",
    base=t["base"].max(),
    note="Q1 : 3 choix max, Q2 : 2 choix max ; comparer l'ordre des causes plus que les niveaux",
)
finish("gc_03_causes")

## 4. Représentation des sujets importants (n° 4)
Q3 (political programmes) and Q4 (media), 100 % stacked bars. "Je ne sais pas" in grey.

In [ ]:
stacked_questions(
    [3, 4],
    {3: "Q3 · Dans les programmes politiques", 4: "Q4 · Dans les médias"},
    "Les sujets importants pour vous sont-ils bien représentés ?",
)
finish("gc_04_representation")

## 5. Rapport au politique (n° 5)
Q7, Q8, Q10, Q11 on the same agreement scale. **Careful**: in Q7, "Oui" means the person has difficulties.

In [ ]:
short_labels = {
    7: "Q7 · Difficultés à comprendre les programmes",
    8: "Q8 · Les responsables prennent en compte mes préoccupations",
    10: "Q10 · Je me sens légitime pour donner mon opinion",
    11: "Q11 · J'ai confiance dans les responsables politiques",
}
stacked_questions(
    [7, 8, 10, 11], short_labels, "Rapport au politique", note="échelle d'accord ; Q7 : « Oui » = difficulté"
)
finish("gc_05_rapport_politique")

## 6. Lecture des programmes et consultation (n° 6)
Two different scales: they are plotted separately.

In [ ]:
stacked_questions([6], {6: "Q6 · A déjà lu des programmes politiques"}, "Lecture des programmes politiques")
finish("gc_06a_lecture")

In [ ]:
stacked_questions([9], {9: "Q9 · Importance de consulter les jeunes"}, "Consulter les jeunes sur les programmes")
finish("gc_06b_consultation")

## 7. Acteurs du changement (n° 7)
Q5 is a ranking: we read the share of people who put the agent in **first position**.
The question offers three agents. The **average rank** (in brackets) only makes sense if all three are ranked and the export keeps the selection order; if a single answer is possible it equals 1.0 everywhere and can be ignored.

In [ ]:
q5 = g.ranking_table(qs)
q5 = q5.assign(label=lambda t: t["option"] + "  (rang moyen " + t["rang moyen"].map("{:.1f}".format) + ")")
ch.bars(q5, "label", "% en 1re position", "IC95 bas", "IC95 haut", title=title_of(5), base=q5["base"].max())
finish("gc_07_acteurs")

## 8. Sources d'information (n° 8)

In [ ]:
multiple_choice_bars(16)
finish("gc_08a_sources")

In [ ]:
single_choice_bars(15)
finish("gc_08b_confiance_info")

## 9. Engagement, freins et leviers (n° 9)
Q20 (obstacles) and Q21 (levers): same colours, so they can be compared.

In [ ]:
single_choice_bars(17)
finish("gc_09a_engage")

In [ ]:
multiple_choice_bars(18)
finish("gc_09b_formes_engagement")

In [ ]:
multiple_choice_bars(20)
finish("gc_09c_freins")

In [ ]:
multiple_choice_bars(21)
finish("gc_09d_leviers")

## 10. Voter est-il utile ? (n° 10)

In [ ]:
stacked_questions([19], {19: "Q19 · Voter est utile pour faire évoluer la société"}, "Utilité du vote")
finish("gc_10_vote_utile")

## 11. Intention de vote (n° 11)
Q23: base limited to the people old enough to vote (see Q22). ⚠ The base is smaller than the total.

In [ ]:
single_choice_bars(22)
finish("gc_11a_age_de_voter")

In [ ]:
vote = g.voting_analysis(d)
ch.bars(
    vote, "réponse Q23", "%", "IC95 bas", "IC95 haut", title=title_of(23), base=vote["base (en âge de voter)"].max()
)
finish("gc_11b_intention_vote")

## 12. Les mots de la question ouverte (n° 12)

In [ ]:
answers, words = g.free_text_words(df, qs)
top = words.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.barh(top["mot"], top["occurrences"], height=0.62, color=ch.BLUE)
for y, v in enumerate(top["occurrences"]):
    ax.text(v + top["occurrences"].max() * 0.01, y, str(v), va="center", fontsize=8.5)
ax.set_xlabel("occurrences")
ch._x_grid(ax)
ch._title(
    ax,
    "Q13 · Les mots les plus fréquents",
    base=len(answers) if len(answers) else None,
    note="à lire avec les réponses brutes",
)
finish("gc_12_mots")

In [ ]:
answers.head(15)  # most frequent answers, after normalisation

## Second temps — comparer deux groupes
A difference visible on the chart may come from chance, especially in a small group: always look at the size of each group (`n`).
Example: "Voter est utile" (Q19) by age bracket.

In [ ]:
col = "Q19"
x = d[["tranche_age", col]].dropna()
x["tranche_age"] = x["tranche_age"].astype(str)
long = x.groupby(["tranche_age", col]).size().rename("n").reset_index()
long["base"] = long.groupby("tranche_age")["n"].transform("sum")  # size of each age bracket
long["pct"] = long["n"] / long["base"]
ch.stacked(
    long,
    "tranche_age",
    col,
    "pct",
    order=scale(19),
    title="Q19 · Voter est utile, selon la tranche d'âge",
    note="comparaison indicative : regarder la taille de chaque groupe",
)
finish("gc_13_q19_par_age")

## Croisements entre questions (A à E)
Each cell crosses **two questions**: we check whether the answer to one changes with the answer to the other.
- rows with fewer than 20 respondents are dropped (`MIN_N`) and named in the subtitle;
- a small gap between two groups may come from chance, especially when the groups are small: look at the `n` of each row;
- ⚠ a link is not a cause: "those who believe voting is useful plan to vote more" does not say which one drives the other.

The crossings with **Q23** (voting intention) cover the people **old enough to vote** (Q22 = Oui).

In [ ]:
# Preparation of the crossings (run once)
old_enough = d["Q22"] == "Oui"  # base of the crossings with Q23
Q23_ORDER = ["Non", "Oui", "Je ne sais pas"]


def cross_stacked(row, column, fig_title, **kw):
    """100 % stacked bars of a crossing; the subtitle gives the base and the dropped groups. Returns the summary."""
    long, r = g.cross(d, row, column, **kw)
    ch.stacked(
        long,
        "row",
        "category",
        "pct",
        order=kw.get("column_order"),
        title=fig_title,
        note=" · ".join(x for x in (f"base : {r['n']} personnes", g.stats_note(r)) if x),
    )
    return r

### A. Intention de vote (Q23) selon l'engagement au quotidien (Q17)
Do people who are engaged day to day plan to vote more? Rows are sorted from the most to the least numerous answering "Oui".

In [ ]:
cross_stacked(
    "Q17",
    "Q23",
    "Q23 · Compte aller voter, selon l'engagement au quotidien (Q17)",
    mask=old_enough,
    column_order=Q23_ORDER,
    sort_by="Oui",
)
finish("gc_x1_vote_par_engagement")

### B. Intention de vote (Q23) selon l'utilité perçue du vote (Q19)
The central crossing: does believing that voting changes something go with the intention to vote?

In [ ]:
cross_stacked(
    "Q19",
    "Q23",
    "Q23 · Compte aller voter, selon « voter est utile » (Q19)",
    mask=old_enough,
    row_order=scale(19),
    column_order=Q23_ORDER,
)
finish("gc_x2_vote_par_utilite")

### C. Intention de vote (Q23) selon la confiance dans les responsables politiques (Q11)
Does distrust turn into abstention?

In [ ]:
cross_stacked(
    "Q11",
    "Q23",
    "Q23 · Compte aller voter, selon la confiance dans les responsables (Q11)",
    mask=old_enough,
    row_order=scale(11),
    column_order=Q23_ORDER,
)
finish("gc_x3_vote_par_confiance")

### D. Sources d'information (Q16) selon la tranche d'âge
One bar per age bracket (light to dark), options sorted by the first bracket.

In [ ]:
t, dropped = g.options_by_profile(df, d, qs, 16, "tranche_age", group_order=g.ORDERS["tranche_age"])
sizes = t.drop_duplicates("groupe").set_index("groupe")["n groupe"]
group_label = {k: f"{k} (n = {v})" for k, v in sizes.items()}
note = f"groupes écartés (< {g.MIN_N}) : {', '.join(dropped)}" if dropped else None
ch.grouped(
    t.assign(groupe=t["groupe"].map(group_label)),
    "option",
    "groupe",
    "pct",
    series_order=[group_label[k] for k in g.ORDERS["tranche_age"] if k in group_label],
    title="Q16 · Sources d'information sur les élections, selon la tranche d'âge",
    base=sizes.sum(),
    note=note,
)
finish("gc_x4_sources_par_age")

### E. Freins (Q20) et leviers (Q21) selon l'intention de vote (Q23)
Do the people who do **not** plan to vote (Q23 = Non) quote different obstacles and levers than those who plan to vote (Q23 = Oui)? "Je ne sais pas" is left aside.
⚠ The "Non" are often few: the subtitle warns when the count falls below 20.

In [ ]:
no_vote = old_enough & (d["Q23"] == "Non")
yes_vote = old_enough & (d["Q23"] == "Oui")
NAME_A, NAME_B = "Ne comptent pas voter (Q23 = Non)", "Comptent voter (Q23 = Oui)"


def compare_and_plot(n, fig_title):
    """Grouped bars of the options ticked by group A vs group B. Returns the table (largest gap first)."""
    r = g.compare_options(df, qs, n, no_vote, yes_vote, NAME_A, NAME_B)
    t = r.melt(id_vars="option", value_vars=[NAME_A, NAME_B], var_name="groupe", value_name="pct")
    na, nb = int(no_vote.sum()), int(yes_vote.sum())
    note = f"n = {na} « Non » contre n = {nb} « Oui »"
    if na < g.MIN_N:
        note += f" · ⚠ moins de {g.MIN_N} « Non » : comparaison fragile"
    ch.grouped(t, "option", "groupe", "pct", series_order=[NAME_A, NAME_B], title=fig_title, base=na + nb, note=note)
    return r

In [ ]:
obstacles = compare_and_plot(20, "Q20 · Freins à la participation, selon l'intention de vote")
finish("gc_x5a_freins_par_vote")

In [ ]:
obstacles[["option", NAME_A, NAME_B, "écart B − A (points)"]]

In [ ]:
levers = compare_and_plot(21, "Q21 · Leviers pour voter, selon l'intention de vote")
finish("gc_x5b_leviers_par_vote")

In [ ]:
levers[["option", NAME_A, NAME_B, "écart B − A (points)"]]